In [1]:
from timeline.api.timeline import Timeline
from timeline.storage.repository import SQLiteRepository
from timeline.domain.enums import DeviceSource
from timeline.domain.models import Event, Chunk
from pathlib import Path
database: Path = Path("../timeline.db")


In [2]:
import polars as pl

In [2]:
repo = SQLiteRepository(database)
timel = Timeline(repo).source(DeviceSource.PHONE)


In [3]:
timel.day_picture("2026-08-03")

DayPicture(day=datetime.date(2026, 8, 3), marker=None, tracked_seconds=86370, active_screen_seconds=23961, sleep_seconds=32717, top_activity='Sport', activity_summary=shape: (11, 8)
┌──────────┬──────────┬──────────┬─────────────┬────────────┬────────────┬────────────┬────────────┐
│ category ┆ name     ┆ sessions ┆ total_secon ┆ average_se ┆ median_sec ┆ minimum_se ┆ maximum_se │
│ ---      ┆ ---      ┆ ---      ┆ ds          ┆ conds      ┆ onds       ┆ conds      ┆ conds      │
│ str      ┆ str      ┆ u32      ┆ ---         ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│          ┆          ┆          ┆ i64         ┆ f64        ┆ f64        ┆ i64        ┆ i64        │
╞══════════╪══════════╪══════════╪═════════════╪════════════╪════════════╪════════════╪════════════╡
│ Body     ┆ Sleep    ┆ 2        ┆ 32717       ┆ 16358.5    ┆ 16358.5    ┆ 2819       ┆ 29898      │
│ Street   ┆ Sport    ┆ 1        ┆ 10632       ┆ 10632.0    ┆ 10632.0    ┆ 10632      ┆ 10632      │
│ Relax   

In [ ]:
def events_from_chunks(chunks: list[Chunk]) -> list[Event]:
    events = []
    for c in chunks:
        events.append(repo.get(Event, c.start_event_id))
        events.append(repo.get(Event, c.end_event_id))
    return events

In [8]:
for i in range(0, 50, 2):
    print(f"[{i}] {len(timel.min_duration(i).chunks())}")

[0] 645
[2] 624
[4] 595
[6] 571
[8] 551
[10] 516
[12] 491
[14] 471
[16] 452
[18] 431
[20] 413
[22] 399
[24] 387
[26] 380
[28] 371
[30] 362
[32] 348
[34] 341
[36] 329
[38] 322
[40] 313
[42] 302
[44] 297
[46] 282
[48] 271


In [ ]:
for c in timel.last(days=1):
    print(f"{c.name}: {c.start_timestamp.time()} for {c.duration_seconds/60:.0f} minutes")

In [19]:
print(timel.metadata.categories)

['Body', 'Home', 'Mind', 'Relax', 'Street']


In [5]:
timel.day("2026-07-29").activity(("Working", "Sleep")).statistics.chunks.summary()

Metric,Value
str,str
"""Sessions""","""6"""
"""Total time""","""10h 44m 5s"""
"""Total hours""","""10.73 h"""
"""Average session""","""1h 47m 21s"""
"""Median session""","""36m 39s"""
"""Shortest session""","""2s"""
"""Longest session""","""7h 45m 26s"""
"""Standard deviation""","""2h 58m 45s"""


In [6]:
df = timel.to_polars()

In [7]:
assert df["duration_seconds"].dtype == pl.Int64
df

id,start_timestamp,end_timestamp,duration_seconds,category,name,source,start_event_id,end_event_id
i64,datetime[μs],datetime[μs],i64,str,str,str,i64,i64
287,2026-07-28 18:20:30,2026-07-28 18:59:10,2320,"""Street""","""Walking""","""embed""",1,2
288,2026-07-28 19:06:02,2026-07-28 19:22:47,1005,"""Street""","""Walking""","""embed""",3,4
289,2026-07-28 19:22:47,2026-07-28 20:23:24,3637,"""Home""","""Working""","""embed""",5,6
290,2026-07-28 20:23:24,2026-07-29 00:06:14,13370,"""Street""","""Social""","""embed""",7,8
291,2026-07-29 00:06:14,2026-07-29 00:06:16,2,"""Body""","""Sleep""","""embed""",9,10
…,…,…,…,…,…,…,…,…
325,2026-07-30 08:33:26,2026-07-30 09:06:33,1987,"""Relax""","""Phone""","""embed""",77,78
326,2026-07-30 09:06:33,2026-07-30 09:13:35,422,"""Home""","""Cooking""","""embed""",79,80
327,2026-07-30 09:13:35,2026-07-30 09:27:12,817,"""Body""","""Eating""","""embed""",81,82


In [8]:
timel.activity("Sleep").chunks()

[Chunk(start_timestamp=datetime.datetime(2026, 7, 29, 0, 6, 14), end_timestamp=datetime.datetime(2026, 7, 29, 0, 6, 16), duration_seconds=2, category='Body', name='Sleep', source=<DeviceSource.EMBED: 'embed'>, start_event_id=9, end_event_id=10, id=291),
 Chunk(start_timestamp=datetime.datetime(2026, 7, 29, 1, 1, 39), end_timestamp=datetime.datetime(2026, 7, 29, 8, 47, 5), duration_seconds=27926, category='Body', name='Sleep', source=<DeviceSource.EMBED: 'embed'>, start_event_id=21, end_event_id=22, id=297),
 Chunk(start_timestamp=datetime.datetime(2026, 7, 30, 1, 8, 27), end_timestamp=datetime.datetime(2026, 7, 30, 8, 16, 50), duration_seconds=25703, category='Body', name='Sleep', source=<DeviceSource.EMBED: 'embed'>, start_event_id=73, end_event_id=74, id=323)]

In [12]:
timel.statistics.chunks.by_day()

day,sessions,total_seconds,average_seconds,median_seconds,minimum_seconds,maximum_seconds
date,u32,i64,f64,f64,i64,i64
2026-07-28,4,20332,5083.0,2978.5,1005,13370
2026-07-29,28,83910,2996.785714,954.5,2,27926
2026-07-30,11,36419,3310.818182,808.0,295,25703


In [9]:
df["end_timestamp"].head(15)

end_timestamp
datetime[μs]
2026-07-28 18:59:10
2026-07-28 19:22:47
2026-07-28 20:23:24
2026-07-29 00:06:14
2026-07-29 00:06:16
…
2026-07-29 08:47:05
2026-07-29 08:59:21
2026-07-29 09:16:34


In [11]:
# Daily totals for an activity
activity_name = "Sleep"

df.filter(pl.col("name") == activity_name).group_by(
    pl.col("start_timestamp").dt.date()
).agg(
    pl.sum("duration_seconds")
).with_columns(
    (
        (pl.col("duration_seconds") // 3600).cast(pl.String)
        + ":"
        + ((pl.col("duration_seconds") % 3600 // 60).cast(pl.String).str.zfill(2))
        + ":"
        + ((pl.col("duration_seconds") % 60).cast(pl.String).str.zfill(2))
    ).alias("duration")
)

start_timestamp,duration_seconds,duration
date,i64,str
2026-07-29,27928,"""7:45:28"""
2026-07-30,25703,"""7:08:23"""


In [12]:
# Monthly totals for activities
(
    timel
    .month("2026-07")
    .to_polars()
    .group_by("name")
    .agg(pl.sum("duration_seconds"))
    .sort("duration_seconds", descending=True)
    .with_columns(
    (
        (pl.col("duration_seconds") // 3600).cast(pl.String)
        + ":"
        + ((pl.col("duration_seconds") % 3600 // 60).cast(pl.String).str.zfill(2))
        + ":"
        + ((pl.col("duration_seconds") % 60).cast(pl.String).str.zfill(2))
    ).alias("duration")
)
)

name,duration_seconds,duration
str,i64,str
"""Sleep""",53631,"""14:53:51"""
"""Social""",23497,"""6:31:37"""
"""Working""",14994,"""4:09:54"""
"""Sport""",14087,"""3:54:47"""
"""Phone""",13258,"""3:40:58"""
"""Eating""",8467,"""2:21:07"""
"""Walking""",5571,"""1:32:51"""
"""Hygiene""",5041,"""1:24:01"""
"""Cooking""",2115,"""0:35:15"""


In [ ]:
# Statistics for all sessions
(
    timel
    .to_polars()
    .group_by("name")
    .agg(
        pl.len().alias("sessions"),
        pl.sum("duration_seconds").alias("total"),
        pl.mean("duration_seconds").alias("average"),
        pl.max("duration_seconds").alias("longest"),
    )
    .sort("total", descending=True)
)

In [ ]:
#Time working per day
(
    timel
    .activity("Working")
    .to_polars()
    .group_by(
        pl.col("start_timestamp").dt.date()
    )
    .agg(
        pl.sum("duration_seconds").alias("seconds")
    )
    .sort(pl.col("start_timestamp"))
)

start_timestamp,seconds
date,i64
2026-07-28,3637
2026-07-29,10717
2026-07-30,600
